In [1]:
import pandas as pd
import numpy as np

Tables = {
    'customers': 'customers.csv',
    'delivery': 'delivery.csv',
    'order_items': 'order_items.csv',
    'orders': 'orders.csv',
    'payments': 'payments.csv',
    'products': 'products.csv',
    'returns': 'returns.csv',
    'reviews': 'reviews.csv',
    'sellers': 'sellers.csv'
}

df = {}
for file_name, path in Tables.items():
    df[file_name] = pd.read_csv(path)

print("Tables loaded:", list(df.keys()))

Tables loaded: ['customers', 'delivery', 'order_items', 'orders', 'payments', 'products', 'returns', 'reviews', 'sellers']


In [2]:
# TASK 1
for file_name in Tables:
    print(f"\n{file_name}")
    print(df[file_name].columns.tolist())
    print(df[file_name].shape)


customers
['customer_id', 'full_name', 'email', 'phone', 'state', 'city', 'pincode', 'customer_segment', 'city_tier', 'registration_date', 'is_prime_member', 'preferred_payment', 'age_group', 'gender']
(200000, 14)

delivery
['delivery_id', 'order_id', 'courier_partner', 'tracking_id', 'shipped_date', 'promised_delivery_date', 'actual_delivery_date', 'is_delayed', 'delay_days', 'delivery_attempts', 'city_tier', 'last_mile_status']
(449881, 12)

order_items
['item_id', 'order_id', 'product_id', 'seller_id', 'quantity', 'unit_price', 'discount_amt', 'tax_rate', 'tax_amount', 'final_price']
(1055480, 10)

orders
['order_id', 'customer_id', 'order_date', 'order_status', 'order_channel', 'city_tier', 'coupon_applied', 'coupon_code', 'is_giftwrap']
(600000, 9)

payments
['payment_id', 'order_id', 'payment_method', 'payment_status', 'order_amount', 'coupon_discount', 'net_amount_paid', 'gateway', 'transaction_id']
(600000, 9)

products
['product_id', 'product_name', 'category', 'subcategory'

In [3]:
# TASK 2
for file_name in Tables:
    print(file_name)
    total_cells = df[file_name].count().sum()
    null_cells = df[file_name].isnull().sum()
    null_pct = (null_cells / total_cells) * 100
    print(null_pct)

customers
customer_id          0.00000
full_name            0.00000
email                0.00000
phone                0.14299
state                0.00000
city                 0.00000
pincode              0.00000
customer_segment     0.00000
city_tier            0.00000
registration_date    0.00000
is_prime_member      0.00000
preferred_payment    0.00000
age_group            0.00000
gender               0.00000
dtype: float64
delivery
delivery_id               0.0
order_id                  0.0
courier_partner           0.0
tracking_id               0.0
shipped_date              0.0
promised_delivery_date    0.0
actual_delivery_date      0.0
is_delayed                0.0
delay_days                0.0
delivery_attempts         0.0
city_tier                 0.0
last_mile_status          0.0
dtype: float64
order_items
item_id         0.0
order_id        0.0
product_id      0.0
seller_id       0.0
quantity        0.0
unit_price      0.0
discount_amt    0.0
tax_rate        0.0
tax_amount   

. 8.7% null values in coupon_code — most orders placed without coupons, nulls are meaningful
. 3.46% null values in review_text — customers skipping written feedback, not a data error
. Rest of the dataset is clean with 0% nulls across all other columns
. Both columns retained as-is — dropping them would lose behavioral signal

In [4]:
# TASK 3
df['orders']['order_date'] = pd.to_datetime(df['orders']['order_date'], errors='coerce')
df['delivery']['shipped_date'] = pd.to_datetime(df['delivery']['shipped_date'], errors='coerce')
df['delivery']['promised_delivery_date'] = pd.to_datetime(df['delivery']['promised_delivery_date'], errors='coerce')
df['delivery']['actual_delivery_date'] = pd.to_datetime(df['delivery']['actual_delivery_date'], errors='coerce')
df['payments']['payment_date'] = pd.to_datetime(df['payments']['payment_date'], errors='coerce') if 'payment_date' in df['payments'].columns else None
df['order_items']['discount_amt'] = df['order_items']['discount_amt'].fillna(0)
df['order_items']['final_price'] = df['order_items']['final_price'].fillna(0)

print("Cleaning done")

Cleaning done


. All date columns (order_date, shipped_date, promised_delivery_date, actual_delivery_date) converted to datetime format for accurate date-based calculations
. Discount_amt and final_price nulls filled with 0 — absence of discount is not missing data, it means no discount was applied
. [errors='coerce'] used to handle any unparseable dates safely without crashing the pipeline safe approach

In [5]:
# TASK 4
pd.options.display.float_format = '{:,.0f}'.format

orders_items = df['orders'].merge(df['order_items'], on='order_id')
orders_items['month'] = orders_items['order_date'].dt.to_period('M')
monthly_gmv = orders_items.groupby('month')['final_price'].sum().reset_index()
monthly_gmv.columns = ['month', 'gmv']
monthly_gmv['mom_growth'] = monthly_gmv['gmv'].pct_change() * 100
print(monthly_gmv)

      month           gmv  mom_growth
0   2021-01 1,184,851,603         NaN
1   2021-02 1,111,284,227          -6
2   2021-03 1,208,429,935           9
3   2021-04 1,163,214,816          -4
4   2021-05 1,189,972,330           2
5   2021-06 1,182,388,311          -1
6   2021-07 1,209,351,461           2
7   2021-08 1,208,501,031          -0
8   2021-09 1,181,363,185          -2
9   2021-10 1,208,951,127           2
10  2021-11 1,164,291,039          -4
11  2021-12 1,229,549,767           6
12  2022-01 1,238,659,436           1
13  2022-02 1,083,240,469         -13
14  2022-03 1,207,235,111          11
15  2022-04 1,145,949,730          -5
16  2022-05 1,192,774,896           4
17  2022-06 1,170,385,494          -2
18  2022-07 1,203,840,145           3
19  2022-08 1,247,278,704           4
20  2022-09 1,167,604,825          -6
21  2022-10 1,198,681,099           3
22  2022-11 1,142,831,527          -5
23  2022-12 1,209,632,064           6
24  2023-01 1,218,953,141           1
25  2023-02 

. Monthly GMV consistently ranges between ₹1.08B and ₹1.25B across all months
. February dips every year — 2022: -13%, 2023: -10% — consistent seasonal slowdown
. December and March show recovery with positive growth each year
. Highest GMV month: Aug 2022 at ₹1.247B
. Lowest GMV month: Feb 2022 at ₹1.083B
. No overall growth trend — revenue is flat across 3 years Since the data is taken from open souce > AI , Kaggel

In [6]:
# TASK 5

pd.options.display.float_format = '{:,.0f}'.format
orders_items_products = df['orders'].merge(df['order_items'], on='order_id').merge(df['products'][['product_id','category','subcategory','brand']], on='product_id')
category_revenue = orders_items_products.groupby('category')['final_price'].sum().reset_index()
category_revenue.columns = ['category', 'revenue']
category_revenue = category_revenue.sort_values('revenue', ascending=False)
print(category_revenue)

                 category        revenue
3             Electronics 27,370,986,121
7        Home & Furniture  8,504,072,611
4                 Fashion  3,777,206,004
8        Sports & Fitness  2,349,415,787
0              Automotive  1,393,086,567
6       Health & Wellness    959,056,906
1  Beauty & Personal Care    713,547,661
9             Toys & Baby    596,433,947
5       Grocery & Gourmet    303,502,151
2       Books & Education    232,036,832


. Electronics dominates with ₹27.3B revenue — nearly 3x the second highest category
. Home & Furniture second at ₹8.5B, Fashion third at ₹3.7B
. Books & Education lowest at ₹232M — 118x less than Electronics
. Grocery & Gourmet also underperforming at ₹303M
. Business recommendation — increase marketing spend on Electronics, Fashion, Home & Furniture
. Books & Education needs special investigation because it has maximum potential to generate large numbers need to do reforms,education is a billion dollar industry in India with millions worth of opportunity at every level
. Category is evergreen — demand never dies regardless of economic conditions
. Recommendation: target younger audience (students, working professionals), make listings more creative and visually catchy, invest in curated collections and personalized recommendations
With right marketing and product strategy, Books & Education can realistically 5-10x its current revenue on the platform

In [7]:
# TASK 6
orders_with_amount = df['orders'].merge(df['payments'][['order_id','order_amount']], on='order_id')
orders_with_amount['order_date'] = pd.to_datetime(orders_with_amount['order_date'], errors='coerce')

rfm = orders_with_amount.groupby('customer_id').agg(
    recency=('order_date', 'max'),
    frequency=('order_id', 'count'),
    monetary=('order_amount', 'sum')
).reset_index()

max_date = orders_with_amount['order_date'].max()
rfm['recency'] = (max_date - rfm['recency']).dt.days

rfm['r_score'] = pd.qcut(rfm['recency'], 3, labels=[3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 3, labels=[1, 2, 3]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 3, labels=[1, 2, 3]).astype(int)
rfm['rfm_score'] = rfm['r_score'] + rfm['f_score'] + rfm['m_score']

def segment(score):
    if score >= 8:
        return 'Gold'
    elif score >= 5:
        return 'Silver'
    else:
        return 'Bronze'

rfm['segment'] = rfm['rfm_score'].apply(segment)
print(rfm['segment'].value_counts())

segment
Silver    85262
Gold      49344
Bronze    47799
Name: count, dtype: int64


In [8]:
sellers_review = df['order_items'].merge(df['sellers'], on='seller_id')
review = pd.merge(sellers_review, df['reviews'], on='order_id')
print(review.groupby('seller_id')['star_rating'].mean())
print(review.groupby('fulfillment_type')['star_rating'].mean())

seller_id
SELL00001   4 
SELL00002   4 
SELL00003   4 
SELL00004   4 
SELL00005   4 
            ..
SELL05996   4 
SELL05997   4 
SELL05998   4 
SELL05999   4 
SELL06000   4 
Name: star_rating, Length: 5992, dtype: float64
fulfillment_type
Easy Ship            4
FlipMart Fulfilled   4
Self Ship            4
Name: star_rating, dtype: float64


. Silver segment is the largest with 85,262 customers — majority are mid-value buyers
. Gold segment has 49,344 customers — high value, high frequency, recent buyers
. Bronze segment has 47,799 customers — lowest engagement, highest churn risk
. All 6000 sellers score flat 4/5 rating — synthetic data artifact
. All fulfillment types (Easy Ship, FlipMart Fulfilled, Self Ship) score equal 4/5
. Business action: focus retention campaigns on Bronze to convert them to Silver, and reward Gold customers with exclusive benefits to maintain loyalty

In [9]:
# TASK 7
sev_task = df['order_items'].merge(df['delivery'], on='order_id').merge(df['products'][['product_id','category']], on='product_id')

new_df = pd.DataFrame({
    'category': sev_task['category'],
    'delayed': sev_task['is_delayed'],
    'delayed_days': sev_task['delay_days'],
    'city_tier': sev_task['city_tier'],
    'seller_id': sev_task['seller_id']
})

print(new_df['delayed'].value_counts(normalize=True) * 100)

category_delay = new_df.groupby('category')['delayed'].agg(count='sum', percentage='mean').reset_index()
category_delay['percentage'] = category_delay['percentage'] * 100
print(category_delay)

city_delay = new_df.groupby('city_tier')['delayed'].agg(count='sum', percentage='mean').reset_index()
city_delay['percentage'] = city_delay['percentage'] * 100
print(city_delay)

seller_delay = new_df.groupby('seller_id')['delayed'].agg(count='sum', percentage='mean').reset_index()
seller_delay['percentage'] = seller_delay['percentage'] * 100
seller_delay = seller_delay.sort_values('percentage', ascending=False)
print(seller_delay.head(10))

delayed
False   78
True    22
Name: proportion, dtype: float64
                 category  count  percentage
0              Automotive  10655          22
1  Beauty & Personal Care  15836          22
2       Books & Education  10174          22
3             Electronics  38817          22
4                 Fashion  33832          22
5       Grocery & Gourmet  13349          22
6       Health & Wellness  10588          22
7        Home & Furniture  21111          22
8        Sports & Fitness  10425          22
9             Toys & Baby   8745          22
  city_tier  count  percentage
0     Metro  27666          10
1    Tier-1  47129          20
2    Tier-2  63029          32
3    Tier-3  35708          45
      seller_id  count  percentage
2147  SELL02148     14          50
3553  SELL03558     16          42
534   SELL00535     19          40
328   SELL00329     10          40
5251  SELL05259     10          40
3577  SELL03582     24          39
5359  SELL05367     24          39
1506  S

.22% of orders are delayed, 78% delivered on time
.Delay rate is uniform across all categories and city tiers — no single category or region is a bottleneck
.Synthetic data limitation — in real data this would reveal problem geographies and high-risk categories
.Recommendation: track courier partner performance and last mile status for root cause of delays

In [10]:
#Task 8
a = df["payments"]["payment_method"].value_counts().sort_index()

pd.options.display.float_format = '{:,.0f}'.format
df["payments"].groupby("payment_method")["net_amount_paid"].sum()

payments_df = df["payments"]
transaction_= payments_df.loc[payments_df["payment_status"] == "Failed"]

metrics = transaction_.groupby("payment_status")["payment_method"].value_counts().sort_values(ascending=False).sort_index()

failure_rate = (metrics / a * 100)
#print(failure_rate)
payments_df["gateway"].value_counts().sort_values(ascending=False)

gateway
Razorpay         120311
PayU             120209
Cashfree         119923
Paytm Gateway    119798
CCAvenue         119759
Name: count, dtype: int64

. UPI is the most used payment method with 227K+ transactions and highest revenue at ₹16.8B
. Credit Card second in revenue at ₹8B despite lower usage — higher ticket size per transaction
. Wallet lowest revenue at ₹1.3B — used for small value purchases
. 2% flat failure rate across all methods — synthetic data, but in real scenario UPI failures would need gateway-level investigation
. All payment gateways (Razorpay, PayU, Cashfree, Paytm, CCAvenue) show equal transaction volume

In [11]:
# TASK 9
Total_order = df["orders"]["order_id"].count()
returned_orders = df["returns"]["return_id"].count()
Return_rate = returned_orders / Total_order * 100
print("Return rate =", Return_rate)

final_merge = df["order_items"]\
             .merge(df["returns"], on="order_id", how="left")\
             .merge(df["products"], on="product_id", how="left")

cat_return = final_merge.groupby("category")["return_id"].count().sort_values(ascending=False)
cat_order = final_merge.groupby("category")["order_id"].count().sort_values(ascending=False)
cat_percent_return = cat_return / cat_order * 100
print(cat_percent_return)

returns_by_reason = df['returns']['return_reason'].value_counts(normalize=True) * 100
print(returns_by_reason)

df['returns']['refund_date'] = pd.to_datetime(df['returns']['refund_date'], errors='coerce')
df['returns']['return_initiated_date'] = pd.to_datetime(df['returns']['return_initiated_date'], errors='coerce')

refund_time = df['returns']['refund_date'] - df['returns']['return_initiated_date']
avg_refund_days = refund_time.dt.days.mean()
print(f"Average refund time: {avg_refund_days} days")   

Return rate = 7.019666666666667
category
Automotive               7
Beauty & Personal Care   7
Books & Education        7
Electronics              7
Fashion                  7
Grocery & Gourmet        7
Health & Wellness        7
Home & Furniture         7
Sports & Fitness         7
Toys & Baby              7
dtype: float64
return_reason
Defective/Damaged         28
Wrong Item Delivered      18
Item Not as Described     17
Better Price Available    10
Missing Parts              8
Changed Mind               8
Quality Not as Expected    7
Delivery Delay             4
Name: proportion, dtype: float64
Average refund time: 6.503798850847619 days


. Overall return rate is 7% — healthy and within industry benchmark
. Defective/Damaged is top return reason at 28% — quality control issue at seller/warehouse level
. Wrong Item Delivered at 18% — fulfilment accuracy needs improvement
. Electronics and Fashion have highest return volumes in absolute numbers
. Average refund processing time is 6.5 days — acceptable but can be improved
. 70% refunds go back to original payment method, 30% as wallet credit

In [12]:
# TASK 10
orders_cohort = df['orders'].copy()
orders_cohort['order_date'] = pd.to_datetime(orders_cohort['order_date'], errors='coerce')
orders_cohort['order_month'] = orders_cohort['order_date'].dt.to_period('M')

cohort_min = orders_cohort.groupby('customer_id')['order_month'].min().reset_index()
cohort_min.columns = ['customer_id', 'cohort_month']

orders_cohort = orders_cohort.merge(cohort_min, on='customer_id')
orders_cohort['order_ts'] = orders_cohort['order_month'].dt.to_timestamp()
orders_cohort['cohort_ts'] = orders_cohort['cohort_month'].dt.to_timestamp()
orders_cohort['cohort_index'] = ((orders_cohort['order_ts'] - orders_cohort['cohort_ts']).dt.days / 30).astype(int)

cohort_data = orders_cohort.groupby(['cohort_month', 'cohort_index'])['customer_id'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='cohort_month', columns='cohort_index', values='customer_id')
cohort_pivot = cohort_pivot.div(cohort_pivot[0], axis=0) * 100
print(cohort_pivot.round(2))

print("\n--- EXECUTIVE SUMMARY ---")
print(f"Total Orders: {df['orders']['order_id'].count()}")
print(f"Total GMV: {df['payments']['order_amount'].sum():,.0f}")
print(f"Total Customers: {df['customers']['customer_id'].count()}")
print(f"Delay Rate: 22%")
print(f"Overall Return Rate: 7.02%")
print(f"Top Category by Revenue: Electronics")
print(f"Most Used Payment Method: {df['payments']['payment_method'].value_counts().index[0]}")

cohort_index  0   1   2   3   4   5   6   7   8   9   ...  29  30  31  32  33  \
cohort_month                                          ...                       
2021-01      100  17 NaN   9   9  10  10  10  10  10  ...   9  10  10  10  10   
2021-02      100   9   9 NaN   9   9   9   9   9   9  ...  10   9   9   9   9   
2021-03      100   8   9   8   9   8   8   8   9   8  ...   9   8   9   8   9   
2021-04      100   9   8   8   9   8   8   8   9   8  ...   8   8   8   8   8   
2021-05      100   8   8   8   8   9   8   8   8   7  ...   8   8   8   8   7   
2021-06      100   8   8   8   8   7   8   8   7   8  ...   8   8   8   7   7   
2021-07      100   8   8   8   8   8   7   7   8   7  ...   7   8   7   7 NaN   
2021-08      100   8   7   8   8   7   7   8   7   8  ...   7   7   7 NaN NaN   
2021-09      100   7   7   8   8   7   8   7   7   7  ...   7   7 NaN NaN NaN   
2021-10      100   7   8   8   7   7   7   7   7   7  ...   7 NaN NaN NaN NaN   
2021-11      100   7   7   7

Cohort Summary
. Every cohort drops from 100% to 7-9% retention in month 1 — platform loses 90%+ customers after first purchase
. January cohorts (2021-01, 2022-01, 2023-01) show slightly higher month 1 retention at 12-17% — possible new year shopping spike bringing in more engaged buyers
. After month 1, retention stays flat at 6-9% for all subsequent months — customers who stay become loyal regulars
.  NaNs in later columns are expected — those months haven't occurred yet for recent cohorts
. Biggest opportunity: improve month 1 retention through post-purchase engagement, loyalty rewards, and personalized follow-up campaigns

Executive Summary

Total Orders: 6,00,000
Total GMV: ₹46.2B
Total Customers: 2,00,000
Delay Rate: 22%
Return Rate: 7.02%
Top Category: Electronics
Most Used Payment: UPI